# Probability of Default (PD) Model & Expected Loss Calculator
# Credit Risk
    Pipeline
    --------
      1. Load & explore the loan data (EDA)
      2. Feature engineering
      3. Train 3 models: Logistic Regression, Random Forest, XGBoost
      4. Evaluate & compare (ROC-AUC, Precision-Recall, KS statistic)
      5. Select best model
      6. expected_loss() — the production-ready function:
           inputs  : loan features
           outputs : PD, Expected Loss  (assuming 10% recovery rate)

In [8]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import xgboost as xgb
import joblib

# Import and understand data

In [9]:
df = pd.read_csv("Loan_Data.csv")
print(f"Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Default rate : {df['default'].mean():.2%}  ({df['default'].sum():,} defaults)")
print(f"Missing vals : {df.isnull().sum().sum()}")
print("\nFeature summary:")
print(df.drop(columns=["customer_id","default"]).describe().round(2).to_string())

Shape        : 10,000 rows × 8 columns
Default rate : 18.51%  (1,851 defaults)
Missing vals : 0

Feature summary:
       credit_lines_outstanding  loan_amt_outstanding  total_debt_outstanding     income  years_employed  fico_score
count                  10000.00              10000.00                10000.00   10000.00        10000.00    10000.00
mean                       1.46               4159.68                 8718.92   70039.90            4.55      637.56
std                        1.74               1421.40                 6627.16   20072.21            1.57       60.66
min                        0.00                 46.78                   31.65    1000.00            0.00      408.00
25%                        0.00               3154.24                 4199.84   56539.87            3.00      597.00
50%                        1.00               4052.38                 6732.41   70085.83            5.00      638.00
75%                        2.00               5052.90              

In [ ]:
# ── EDA plots ────────────────────────────────────────────────
features = ["credit_lines_outstanding","loan_amt_outstanding",
            "total_debt_outstanding","income","years_employed","fico_score"]

fig = plt.figure(figsize=(16, 14))
fig.suptitle("Loan Data — Exploratory Analysis", fontsize=14, fontweight="bold")
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)